<a href="https://colab.research.google.com/github/yabswannalearn/Fine-Tuned-Translator/blob/main/Eng_to_Kamp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install evaluate sacrebleu

## Raw model

In [54]:
from unsloth import FastLanguageModel
import torch

fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",

    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit"
]

## the model that we will fine tune
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "YOUR_HF_TOKEN",      # HF Token for gated models
)

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [84]:
## raw model for testing
raw_model, raw_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
)

FastLanguageModel.for_inference(raw_model)

==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151654)
    (layers): ModuleList(
      (0-1): 2 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear4bit(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
      

## Testing Loaded Model

In [97]:
# Enable native 2x faster inference
FastLanguageModel.for_inference(raw_model)

prompt_template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Translate the following Kapampangan sentence to English {}

### Response:
{}"""

# Tokenize the prompt with the Kapampangan input
inputs = tokenizer(
[
    prompt_template.format(
        "mengan naka?", # Input text
        "", # Leave the response blank for the model to generate
    )
], return_tensors = "pt").to("cuda")

# Generate and print the base model's prediction
outputs = raw_model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Translate the following Kapampangan sentence to English mengan naka?

### Response:
The user's query is in Kapampangan, and the response should be in English. The instruction is to translate the sentence "mengan naka?" into English. The correct translation is "What is your name?" or "What is your name?" in English. The response should be concise and accurate.

### Final


## Initiate QLoRa for Fine Tuning

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules= ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42
)

## Setting up the dataset

In [57]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("Coco-18/Kapampangan-English", split = "train")
dataset = dataset.rename_columns({"target": "kapampangan", "source": "english"})

# Prompt template
prompt_template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Translate the following Kapampangan sentence to English: {}

### Response:
{}"""

# Get the End Of Sequence (EOS) token
EOS_TOKEN = tokenizer.eos_token

# Create a function to format the dataset
def formatting_prompts_func(examples):
    inputs  = examples["kapampangan"] # kapampangan
    outputs = examples["english"] # english
    texts = []
    for pam, eng in zip(inputs, outputs):
        # Inject the Kapampangan and English pairs into the template, then add the EOS token
        text = prompt_template.format(pam, eng) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Apply the formatting
formatted_dataset = dataset.map(formatting_prompts_func, batched = True)

## Fine Tuning the Model

In [58]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 60, # Note: 60 is just a quick test. Increase this to 500+ or use num_train_epochs for a full run.
        num_train_epochs = 5, # Note: 1 is just a quick test. Increase this to 5+ for a full run.
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Disables external logging trackers
    ),
)

# This starts the actual fine-tuning process
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,217 | Num Epochs = 5 | Total steps = 695
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 34,865,152 of 1,755,440,128 (1.99% trained)


Step,Training Loss
1,4.739700
2,4.843000
3,4.576000
4,4.112000
5,3.707900
6,2.992300
7,2.763200
8,2.836000
9,2.378100
10,2.213600


## Inference on the fine tuned model

In [94]:
# Enable native 2x faster inference
FastLanguageModel.for_inference(model)


inputs = tokenizer(
[
    prompt_template.format(
        "mengan naka?", # Input text
        "", # Leave the response blank for the model to generate
    )
], return_tensors = "pt").to("cuda")

# Generate and print the fine-tuned model's prediction
outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Translate the following Kapampangan sentence to English, only output the response: mengan naka?

### Response:
have you eaten?


## Benchmarking Bleu and Chrf and a side by side comparison of the raw model

In [83]:
import evaluate
from tqdm import tqdm

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

test_kapampangan = dataset["kapampangan"][:50]
reference_english = dataset["english"][:50]

raw_predictions = []
tuned_predictions = []

FastLanguageModel.for_inference(raw_model)
FastLanguageModel.for_inference(model)

for pam_text in tqdm(test_kapampangan, desc="Translating (Both Models)"):
    # Format the prompt
    prompt = prompt_template.format(pam_text, "")

    # --- RAW MODEL GENERATION ---
    raw_inputs = raw_tokenizer([prompt], return_tensors="pt").to("cuda")
    raw_outputs = raw_model.generate(**raw_inputs, max_new_tokens=64, use_cache=True)
    raw_full_output = raw_tokenizer.batch_decode(raw_outputs, skip_special_tokens=True)[0]
    raw_translation = raw_full_output.split("### Response:\n")[-1].strip()
    raw_predictions.append(raw_translation)

    # --- TUNED MODEL GENERATION ---
    tuned_inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    tuned_outputs = model.generate(**tuned_inputs, max_new_tokens=64, use_cache=True)
    tuned_full_output = tokenizer.batch_decode(tuned_outputs, skip_special_tokens=True)[0]
    tuned_translation = tuned_full_output.split("### Response:\n")[-1].strip()
    tuned_predictions.append(tuned_translation)

# Format references for the evaluate library
formatted_references = [[ref] for ref in reference_english]

# Calculate scores for RAW model
raw_bleu_score = bleu.compute(predictions=raw_predictions, references=formatted_references)
raw_chrf_score = chrf.compute(predictions=raw_predictions, references=formatted_references)

# Calculate scores for TUNED model
tuned_bleu_score = bleu.compute(predictions=tuned_predictions, references=formatted_references)
tuned_chrf_score = chrf.compute(predictions=tuned_predictions, references=formatted_references)

# Print the side-by-side comparison
print("\n" + "="*50)
print(" BENCHMARK RESULTS: RAW VS FINE-TUNED")
print("="*50)
print(f"RAW MODEL   - BLEU: {raw_bleu_score['score']:5.2f} | chrF: {raw_chrf_score['score']:5.2f}")
print(f"TUNED MODEL - BLEU: {tuned_bleu_score['score']:5.2f} | chrF: {tuned_chrf_score['score']:5.2f}")
print("="*50)

Translating (Both Models): 100%|██████████| 50/50 [05:47<00:00,  6.95s/it]


 BENCHMARK RESULTS: RAW VS FINE-TUNED
RAW MODEL   - BLEU:  0.03 | chrF:  4.61
TUNED MODEL - BLEU: 72.01 | chrF: 69.78



## Save to Drive

In [73]:
from google.colab import drive
drive.mount('/content/drive')

# Then save to a path in your drive:
model.save_pretrained("/content/drive/MyDrive/kapampangan_model")
tokenizer.save_pretrained("/content/drive/MyDrive/kapampangan_model")

Mounted at /content/drive


('/content/drive/MyDrive/kapampangan_model/tokenizer_config.json',
 '/content/drive/MyDrive/kapampangan_model/special_tokens_map.json',
 '/content/drive/MyDrive/kapampangan_model/chat_template.jinja',
 '/content/drive/MyDrive/kapampangan_model/vocab.json',
 '/content/drive/MyDrive/kapampangan_model/merges.txt',
 '/content/drive/MyDrive/kapampangan_model/added_tokens.json',
 '/content/drive/MyDrive/kapampangan_model/tokenizer.json')